# StoryForge trên Google Colab — git flow

## Chuẩn bị (1 lần)

1. **Runtime**: Change runtime type → **A100** (tốt nhất) / **L4** / **T4 + High-RAM**.
2. **Secrets** (🔑 menu trái):
   - `SF_LLM_API_KEY` — key llmgate
   - `SF_GH_TOKEN` — GitHub PAT (repo `DaniYLab/audio` là **private**, PAT cần quyền `repo`)
3. **Runtime → Run all**.

In [ ]:
# §1 — Kiểm tra môi trường
import torch, subprocess
print('GPU:', torch.cuda.get_device_name(0), '| capability:', torch.cuda.get_device_capability())
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('bf16 supported:', torch.cuda.is_bf16_supported())
with open('/proc/meminfo') as f:
    print([l.strip() for l in f if l.startswith('MemTotal')][0])
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
# §2 — Clone repo (private → dùng SF_GH_TOKEN)
%cd /content
import subprocess, pathlib, shutil
from google.colab import userdata
try:
    gh = userdata.get('SF_GH_TOKEN')
except Exception:
    gh = ''
url = 'https://github.com/DaniYLab/audio.git'
auth_url = f'https://{gh}@github.com/DaniYLab/audio.git' if gh else url
base = pathlib.Path('/content/storyforge')
if base.exists():
    shutil.rmtree(base)
subprocess.run(['git', 'clone', '--depth', '1', '-b', 'feat/completion-sprint', auth_url, str(base)], check=True)
subprocess.run(['git', '-C', str(base), 'remote', 'set-url', 'origin', url], check=True)
print('cloned:', sorted(p.name for p in base.iterdir())[:8])

In [ ]:
# §3 — Giải nén bundle dữ liệu (transcript, story, ảnh, KB) vào repo
import zipfile, pathlib
base = pathlib.Path('/content/storyforge')
with zipfile.ZipFile(base/'colab/luu_lac_bundle.zip') as z:
    z.extractall(base)
print('bundle extracted; workspace:', [p.name for p in (base/'data/workspace/luu_lac_s01e01').iterdir()])

In [ ]:
# §4 — Cài dependencies (~5 phút; KHÔNG đụng torch có sẵn của Colab)
!apt-get -qq install -y ffmpeg deno 2>/dev/null | tail -1
!pip install -q --no-deps -e /content/storyforge
!pip install -q pydantic-settings typer rich pyyaml structlog tenacity httpx yt-dlp \
    qdrant-client FlagEmbedding 'huggingface-hub>=0.34,<1.0' 'diffusers==0.35.2' \
    accelerate imageio imageio-ffmpeg av edge-tts
import storyforge, qdrant_client, diffusers, edge_tts, yt_dlp
print('deps OK | diffusers', diffusers.__version__)

In [ ]:
# §5 — Ghi .env (secret + Qdrant local + Wan dtype theo GPU)
import torch, pathlib
from google.colab import userdata
try:
    llm_key = userdata.get('SF_LLM_API_KEY')
except Exception:
    llm_key = input('Dán SF__LLM__API_KEY: ')
dtype = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
base = pathlib.Path('/content/storyforge')
env = f'''\
SF__WORKSPACE_DIR=data/workspace
SF__LOG_LEVEL=INFO
SF__LOG_FORMAT=console
SF__LLM__BASE_URL=https://llmgate.app/v1
SF__LLM__API_KEY={llm_key}
SF__LLM__WRITER_MODEL=gemini-3.8-flash
SF__LLM__REVIEWER_MODEL=gemini-3.8-flash
SF__KNOWLEDGE__STORE=qdrant
SF__KNOWLEDGE__QDRANT_PATH=data/qdrant_local
SF__KNOWLEDGE__EMBEDDING__PROVIDER=bge_m3_local
SF__KNOWLEDGE__EMBEDDING__MODEL=BAAI/bge-m3
SF__TTS__ENGINE=edge
SF__TTS__EDGE_VOICE=vi-VN-NamMinhNeural
SF__IMAGING__PROVIDER=openai
SF__IMAGING__OPENAI_MODEL=gpt-image-2
SF__VIDEO__FFMPEG_BIN=ffmpeg
SF__VIDEO__FFPROBE_BIN=ffprobe
SF__ANIMATION__PROVIDER=wan_local
SF__ANIMATION__WAN_MODEL=yetter-ai/Wan2.2-TI2V-5B-Turbo-Diffusers
SF__ANIMATION__WAN_STEPS=4
SF__ANIMATION__WAN_GUIDANCE=1.0
SF__ANIMATION__WAN_DTYPE={dtype}
'''
(base/'.env').write_text(env, encoding='utf-8')
print(f'.env written (wan_dtype={dtype})')

In [ ]:
# §6 — Cache model trên Drive (chạy 1 lần; lần sau chỉ mất vài phút)
import os, pathlib
from google.colab import drive
drive.mount('/content/drive')
os.environ['HF_HOME'] = '/content/drive/MyDrive/storyforge/hf_cache'
pathlib.Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
!hf download yetter-ai/Wan2.2-TI2V-5B-Turbo-Diffusers --max-workers 4 | tail -2
!hf download BAAI/bge-m3 | tail -2

In [ ]:
# §7 — Chạy animation + video (nền; sống sót qua timeout cell)
%cd /content/storyforge
!nohup storyforge run --project luu_lac_s01e01 \
    --source-config config/story_config.luu_lac.yaml \
    --only story --only tts --only imaging --only video \
    > data/colab_run.log 2>&1 &
print('đang chạy nền — theo dõi bằng §8')

In [ ]:
# §8 — Theo dõi (chạy lại bất cứ lúc nào)
import pathlib
root = pathlib.Path('/content/storyforge/data/workspace/luu_lac_s01e01')
anim = sorted((root/'06_animation').glob('*.mp4')) if (root/'06_animation').exists() else []
print(f'animation clips: {len(anim)}/10')
log = pathlib.Path('/content/storyforge/data/colab_run.log')
if log.exists():
    print('--- log tail ---')
    print('\n'.join(log.read_text(encoding='utf-8', errors='replace').splitlines()[-8:]))
final = root/'07_video/final.mp4'
if final.exists():
    print(f'✅ VIDEO: {final} ({final.stat().st_size/1e6:.0f} MB)')

In [ ]:
# §9 — Copy kết quả về Drive
import shutil, pathlib
src = pathlib.Path('/content/storyforge/data/workspace/luu_lac_s01e01')
dst = pathlib.Path('/content/drive/MyDrive/storyforge/luu_lac_result')
dst.mkdir(parents=True, exist_ok=True)
for d in ['06_animation', '07_video', '06_images']:
    if (src/d).exists():
        shutil.copytree(src/d, dst/d, dirs_exist_ok=True)
shutil.copy2(src/'04_story/story.json', dst/'story.json')
print('✅ đã copy về Drive/MyDrive/storyforge/luu_lac_result/')